In [0]:
# Silver Layer - Data cleaning and validations
# Reading from Bronze Delta table, applies transformations and saves to Silver

In [0]:
# Configuration
storage_account_name = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_NAME")
storage_account_key = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_KEY")
container_name = dbutils.secrets.get(scope="etl_sales_scope", key="AZURE_CONTAINER_NAME")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
bronze_path = f"{base_path}/bronze/sales"
silver_path = f"{base_path}/silver/sales"

print("Configuration loaded successfully")

In [0]:
# Read Bronze layer

from pyspark.sql.functions import (
    col, to_date, to_timestamp, trim, when, 
    current_timestamp, isnan
)
from pyspark.sql.types import DoubleType, IntegerType

df_bronze = spark.read.format("delta").load(bronze_path)

print(f"Bronze table loaded: {df_bronze.count()} rows")
display(df_bronze)


In [0]:
# Step 1: Rename columns

print("Step 1: Renaming columns...")

df_silver = df_bronze.toDF(
    'order_id',
    'order_date',
    'product_id',
    'product_category',
    'price',
    'discount_percent',
    'quantity_sold',
    'customer_region',
    'payment_method',
    'rating',
    'review_count',
    'discounted_price',
    'total_revenue',
    '_ingested_at',
    '_source_file'
)

print("Columns renamed successfully")


In [0]:
# Step 2: Data type conversion

print("Step 2: Data type conversion...")

# Convert date
df_silver = df_silver.withColumn(
    'order_date', to_date(col('order_date'), 'yyyy-MM-dd')
)

# Verify invalid dates
invalid_dates = df_silver.filter(col('order_date').isNull()).count()
if invalid_dates > 0:
    print(f"⚠️ {invalid_dates} invalid dates found, dropping rows...")
    df_silver = df_silver.filter(col('order_date').isNotNull())

# Convert numerical columns
numerical_columns = {
    'price': DoubleType(),
    'discount_percent': DoubleType(),
    'quantity_sold': IntegerType(),
    'rating': DoubleType(),
    'review_count': IntegerType(),
    'discounted_price': DoubleType(),
    'total_revenue': DoubleType()
}

for column, dtype in numerical_columns.items():
    df_silver = df_silver.withColumn(column, col(column).cast(dtype))

print("Data types converted successfully")

In [0]:
# Step 3: Data cleaning

print("Step 3: Data cleaning...")

# Remove spaces from string columns
string_columns = ['product_category', 'customer_region', 'payment_method']

for column in string_columns:
    df_silver = df_silver.withColumn(column, trim(col(column)))

# Replace empty strings with null
for column in string_columns:
    df_silver = df_silver.withColumn(
        column,
        when(col(column) == '', None).otherwise(col(column))
    )

# Verify null values
print("Null values per column:")
for column in df_silver.columns:
    null_count = df_silver.filter(col(column).isNull()).count()
    if null_count > 0:
        print(f"⚠️ {column}: {null_count} nulls")

# Drop critical columns nulls
critical_columns = [
    'order_id', 'product_id', 'price', 'total_revenue',
    'product_category', 'customer_region', 'payment_method', 'quantity_sold'
]

before = df_silver.count()
df_silver = df_silver.dropna(subset=critical_columns)
after = df_silver.count()
print(f"Dropped {before - after} rows with null critical values")

# Fill non-critical columns
df_silver = df_silver \
    .fillna({'rating': 0.0}) \
    .fillna({'review_count': 0}) \
    .fillna({'discount_percent': 0.0}) \
    .fillna({'discounted_price': 0.0})

print("Null values handled successfully")